In [15]:
%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [1]:
import sys; sys.path.append("..")
%load_ext autoreload
%autoreload 2
import gc, os, time, json
import pandas as pd, lightgbm as lgb, psutil
from src.blocking import build_index, prepare_pool

H, M = "../dataset/hold/", "../models/dev/"
s1_hold = pd.read_csv(H + "s1_hold.tsv", sep="\t")
gt_hold = pd.read_csv(H + "gt_hold.tsv", sep="\t")
pool_df = pd.concat([pd.read_csv(H + "s2_hold.tsv", sep="\t"),
                     pd.read_csv(H + "s3_hold.tsv", sep="\t")], ignore_index=True)

# WHY truth as sets: same format the scorer already uses
truth = {s: (set(m.split(",")) if isinstance(m, str) else set())
         for s, m in zip(gt_hold["source1_entity_id"], gt_hold["matched_entity_ids"])}

proc = psutil.Process(os.getpid())
gb = lambda b: round(b / 1e9, 2)

# WHY gc first: so the "before" number is not inflated by garbage
gc.collect()
r0 = proc.memory_info().rss
t0 = time.time()
index = build_index(pool_df)
r1 = proc.memory_info().rss
prepared = prepare_pool(pool_df)
r2 = proc.memory_info().rss

print("pool rows:", len(pool_df), "(expect 1,402,291) | keys:", len(index))
print("seconds:", round(time.time() - t0, 1))
print("RAM before / after index / after prepared (GB):", gb(r0), gb(r1), gb(r2))
print("free RAM now (GB):", gb(psutil.virtual_memory().available))
# WHY this scale: the full pool is 10,320,219 records
print("rough full-scale index + prepared (GB):", round((r2 - r0) * 10_320_219 / len(pool_df) / 1e9, 1))

pool rows: 1402291 (expect 1,402,291) | keys: 5240781
seconds: 74.4
RAM before / after index / after prepared (GB): 0.95 2.08 3.87
free RAM now (GB): 3.62
rough full-scale index + prepared (GB): 21.5


In [2]:
import lightgbm as lgb, json
from src.pipeline import run_chunked

# WHY the saved config: the features and threshold are frozen, never retuned on this set
cfg = json.load(open(M + "matcher_dev_v4_config.json"))
booster = lgb.Booster(model_file=M + "matcher_dev_v4_lgbm.txt")

t0 = time.time()
pred = run_chunked(s1_hold, index, prepared, booster, cfg, chunk=20_000)
print("total minutes:", round((time.time() - t0) / 60, 1))

# WHY pickle: the run is long, so keep the result and score it again without re-running
pd.to_pickle(pred, H + "pred_hold_v4.pkl")

entities 0-20,000 | 38s | kept rows so far 66,129
entities 20,000-40,000 | 38s | kept rows so far 132,304
entities 40,000-60,000 | 38s | kept rows so far 198,496
entities 60,000-80,000 | 38s | kept rows so far 264,398
entities 80,000-100,000 | 37s | kept rows so far 330,171
entities 100,000-120,000 | 38s | kept rows so far 396,269
entities 120,000-140,000 | 39s | kept rows so far 462,320
entities 140,000-160,000 | 38s | kept rows so far 528,936
entities 160,000-180,000 | 37s | kept rows so far 594,803
entities 180,000-200,000 | 37s | kept rows so far 661,275
entities 200,000-220,000 | 37s | kept rows so far 727,482
entities 220,000-240,000 | 37s | kept rows so far 793,952
entities 240,000-260,000 | 37s | kept rows so far 860,195
entities 260,000-280,000 | 37s | kept rows so far 926,205
entities 280,000-300,000 | 37s | kept rows so far 992,410
total minutes: 9.5


In [3]:
from src.matcher import macro_f05

s1_ids = list(s1_hold["entity_id"])
print("macro F0.5:", round(macro_f05(pred, truth, s1_ids), 4), "(dev validation: 0.9608)")

# WHY pooled precision and recall: same definition as Day 4, so the numbers are comparable
tp = n_pred = n_true = 0
single = single_false = 0
for e in s1_ids:
    p, t = pred.get(e, set()), truth[e]
    if not t:
        single += 1
        single_false += bool(p)
        continue
    tp += len(p & t); n_pred += len(p); n_true += len(t)
print("precision:", round(tp / n_pred, 4), "| recall:", round(tp / n_true, 4))
print("singletons falsely matched:", single_false, "of", single)

# WHY by country: the open item from Day 5, and France will behave differently
country = dict(zip(s1_hold["entity_id"], s1_hold["country"]))
for c in ["US", "India"]:
    ids = [e for e in s1_ids if country[e] == c]
    print(c, round(macro_f05(pred, truth, ids), 4))

macro F0.5: 0.9459 (dev validation: 0.9608)
precision: 0.9764 | recall: 0.9165
singletons falsely matched: 2263 of 16747
US 0.9566
India 0.9298


In [16]:
import pandas as pd

H = "../dataset/hold/"
pred = pd.read_pickle(H + "pred_hold_v4.pkl")
s1_hold = pd.read_csv(H + "s1_hold.tsv", sep="\t")
gt_hold = pd.read_csv(H + "gt_hold.tsv", sep="\t")
pool_df = pd.concat([pd.read_csv(H + "s2_hold.tsv", sep="\t"),
                     pd.read_csv(H + "s3_hold.tsv", sep="\t")], ignore_index=True)

# WHY set_index: lets us read a record's text by its ID
s1_idx = s1_hold.set_index("entity_id")
pool_idx = pool_df.set_index("entity_id")

# WHY: a singleton has no true matches, so every prediction for it is wrong
singles = set(gt_hold.loc[gt_hold["matched_entity_ids"].isna(), "source1_entity_id"])
bad = [e for e in singles if pred.get(e)]

# WHY by country: tells us if the singleton problem is mostly India
print(s1_idx.loc[bad, "country"].value_counts().to_dict())

# WHY random_state: the same examples on every run, so we can discuss them
for e in pd.Series(bad).sample(8, random_state=1):
    a = s1_idx.loc[e]
    print("S1  :", a["business_name"], "|", a["business_address"])
    for c in list(pred[e])[:2]:
        b = pool_idx.loc[c]
        print("  MATCHED:", b["business_name"], "|", b["business_address"])
    print()

{'US': 1206, 'India': 1057}
S1  : Dium | 874 County Street 2921, Tuttle, OK
  MATCHED: Dium (Co) | 876 County Street 2921, Tuttle CITY, Oklahoma

S1  : Asha Ventures Private Limited | 99, Old Prabhadevi Road, Mumbai, Maharashtra
  MATCHED: माय प्रॉपर्टीज प्राइवेट लिमिटेड | 99, MH, Mumbai

S1  : Ready Automobiles Private Limited | Comm-7Pl-191To197-199-206, Sec-10 Prajapatiornpanvel, Panvel, Raigarh(Mh), Maharashtra
  MATCHED: Ready Automobiles Industries [Private] | #667 Comm-7pl-191to197-199-210, Sec-10 Prajapatiornpanvel, Panvel, Raigarh(mh), MH

S1  : Physical Therapy Choice Care Associates LLC | 14415 Hillgrove Court, Milwaukie, OR
  MATCHED: Inc. Physical Therapy Choice Care Associates | 14424 Hillgrove Court, Milwaukie, Oregon

S1  : Ganesh & Sons Limited | 57, Shivshakti Nagar, Indore, Madhya Pradesh
  MATCHED: पायोनियर वेंचर्स प्राइवेट लिमिटेड | 57., Indore, मध्य प्रदेश

S1  : 2401 Third Place Company LLC | 1 Rolling Hills Road, Colonie, NY
  MATCHED: 2401 Third Place Company C

In [17]:
from collections import Counter
from src.normalize import norm_name, norm_address

buckets, sizes = Counter(), []
for e in bad:
    a = s1_idx.loc[e]
    for c in pred[e]:
        b = pool_idx.loc[c]
        n1, n2 = norm_name(a["business_name"]), norm_name(b["business_name"])
        # WHY three buckets: each one needs a different fix
        if n2 == "":
            kind = "record name is native script"
        elif n1 == n2:
            kind = "same cleaned name (decoy copy)"
        else:
            kind = "different name"
        buckets[kind] += 1
        # WHY min size: it is the shorter address that limits how much evidence there is
        sizes.append(min(len(norm_address(a["business_address"])),
                         len(norm_address(b["business_address"]))))

total = sum(buckets.values())
for k, v in buckets.most_common():
    print(f"{k}: {v} ({v/total:.1%})")
# WHY: compare with the typical address size to see if the wrong matches really have short addresses
print("median shorter-address tokens in wrong matches:", pd.Series(sizes).median())

different name: 1472 (59.8%)
same cleaned name (decoy copy): 691 (28.1%)
record name is native script: 297 (12.1%)
median shorter-address tokens in wrong matches: 6.0


In [18]:
import numpy as np
from rapidfuzz import fuzz

rows = []
for e in bad:
    a = s1_idx.loc[e]
    n1, a1 = norm_name(a["business_name"]), norm_address(a["business_address"])
    for c in pred[e]:
        b = pool_idx.loc[c]
        n2, a2 = norm_name(b["business_name"]), norm_address(b["business_address"])
        if n2 == "" or n1 == n2:
            continue          # WHY skip: we only study the "different name" bucket
        ns = fuzz.token_set_ratio(n1, n2) / 100
        aj = len(a1 & a2) / len(a1 | a2) if a1 and a2 else np.nan
        # WHY numbers as sets: same rule as num_match / num_conflict in features.py
        x = {t.lstrip("0") or "0" for t in a1 if t.isdigit()}
        y = {t.lstrip("0") or "0" for t in a2 if t.isdigit()}
        rows.append((e, c, ns, aj, bool(x & y), bool(x) and bool(y) and not (x & y)))

d = pd.DataFrame(rows, columns=["s1", "cand", "name_set", "addr_jac", "num_match", "num_conflict"])
print(len(d))
# WHY quantiles: they show if the names are close or unrelated
print(d[["name_set", "addr_jac"]].describe(percentiles=[.1, .5, .9]).round(2))
print("house number shared:", round(d["num_match"].mean(), 3),
      "| house number conflict:", round(d["num_conflict"].mean(), 3))

# WHY lowest name similarity: these are the "unrelated name" cases
for r in d.sort_values("name_set").head(5).itertuples():
    print(f"\nname_set={r.name_set:.2f} addr_jac={r.addr_jac:.2f}")
    print("S1  :", s1_idx.loc[r.s1, "business_name"], "|", s1_idx.loc[r.s1, "business_address"])
    print("MATCHED:", pool_idx.loc[r.cand, "business_name"], "|", pool_idx.loc[r.cand, "business_address"])

1472
       name_set  addr_jac
count   1472.00   1445.00
mean       0.88      0.64
std        0.18      0.18
min        0.06      0.00
10%        0.62      0.38
50%        1.00      0.67
90%        1.00      0.85
max        1.00      1.00
house number shared: 0.489 | house number conflict: 0.363

name_set=0.06 addr_jac=1.00
S1  : National Integrated Rithm L.L.C. | 220 Marie Antoinette Drive, Hempstead, TX
MATCHED: 5fn  LLC | 220 Marie Antoinette Dr, Hempstead, Texas

name_set=0.07 addr_jac=1.00
S1  : Misti's Hair Studio | 71 Vineyard Drive, Pueblo West, CO
MATCHED: Xyloflux | <NULL>, PUEBLO WEST, CO, 71. VINEYARD DR

name_set=0.07 addr_jac=1.00
S1  : Golden Technologies Limited | Jaipur, Plot No 01, Tara Nagar, Rajasthan, Jaipur
MATCHED: Yumavera | Plot No 01, Tara Nagar, Jaipur, Rajasthan

name_set=0.10 addr_jac=1.00
S1  : Eastern Properties Limited | 404, Nr. Silver Oak Club, Ahmadabad City, Gujarat, Moneyplant Business Hub, Ahmedabad
MATCHED: MS | Moneyplant Business Hub, 404, Nr. S

In [19]:
import numpy as np, json, time
import lightgbm as lgb
from src.pipeline import build_pairs
from src.features import add_relative_features
from src.matcher import train_matcher, macro_f05

cfg = json.load(open(M + "matcher_dev_v4_config.json"))
booster = lgb.Booster(model_file=M + "matcher_dev_v4_lgbm.txt")
F = cfg["features"]

parts, t0 = [], time.time()
for start in range(0, len(s1_hold), 20_000):
    ch = s1_hold.iloc[start:start + 20_000]
    pr = build_pairs(ch, index, prepared)
    # WHY label here: build_pairs has none, and only this experiment needs it
    pr["label"] = [int(c in truth[s]) for s, c in zip(pr["s1"], pr["cand"])]
    pr = add_relative_features(pr)
    # WHY float32: halves the RAM of a table with millions of rows
    num = pr.columns.difference(["s1", "cand", "label"])
    pr[num] = pr[num].astype("float32")
    parts.append(pr)
    print(start, round(time.time() - t0), "s")

pairs = pd.concat(parts, ignore_index=True)
del parts; gc.collect()
pairs.to_pickle(H + "pairs_hold_v4.pkl")   # WHY: rebuilding costs about 10 minutes

# WHY split by S1 entity: first 200k train, last 100k eval, no entity on both sides
train_s1 = set(s1_hold["entity_id"].iloc[:200_000])
eval_ids = list(s1_hold["entity_id"].iloc[200_000:])
tr = pairs[pairs["s1"].isin(train_s1)]
ev = pairs[~pairs["s1"].isin(train_s1)].copy()

def score(p, t=0.70):
    df = ev[["s1", "cand"]].copy()
    df["p"] = p
    # WHY same steps as predict_matches: threshold, then one S1 per record
    df = df[df["p"] >= t].sort_values("p", ascending=False).drop_duplicates("cand")
    pred = df.groupby("s1")["cand"].apply(set).to_dict()
    single = [e for e in eval_ids if not truth[e]]
    false_single = sum(bool(pred.get(e)) for e in single)
    return round(macro_f05(pred, truth, eval_ids), 4), false_single, len(single)

print("frozen v4      :", score(booster.predict(ev[F])))
model_h = train_matcher(tr, F)
for t in [0.6, 0.7, 0.8]:
    print(f"retrained t={t}:", score(model_h.predict_proba(ev[F])[:, 1], t))

0 44 s
20000 85 s
40000 127 s
60000 169 s
80000 213 s
100000 255 s
120000 298 s
140000 339 s
160000 379 s
180000 419 s
200000 458 s
220000 498 s
240000 541 s
260000 583 s
280000 625 s
frozen v4      : (0.9405, 872, 5565)
retrained t=0.6: (0.9452, 676, 5565)
retrained t=0.7: (0.9445, 555, 5565)
retrained t=0.8: (0.9413, 407, 5565)


In [20]:
import json
from pathlib import Path

OUT = Path("../models/hold")
OUT.mkdir(parents=True, exist_ok=True)

# WHY "hold200k" in the name: the file alone says it was trained on hold-out entities
model_h.booster_.save_model(str(OUT / "matcher_hold200k_lgbm.txt"))

# WHY record the scope: this model's entities are not clean for testing any more
json.dump({
    "trained_on": "first 200,000 of the 300k hold-out entities, pool 1,402,291, 22 features",
    "eval_on": "last 100,000 hold-out entities",
    "eval_macro_f05": {"t0.6": 0.9452, "t0.7": 0.9445, "t0.8": 0.9413},
    "frozen_v4_same_eval": 0.9405,
    "features": F,
    "note": "experiment only; the final model is retrained on full-scale pairs (Day 6, Kaggle)",
}, open(OUT / "matcher_hold200k_config.json", "w"), indent=2)
print("saved")

saved
